# Constant frequencies


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import constant_frequencies as cf
from dataloader import load_survey
from paths import FIGURE_DIR
from signal_processing_tools import FS, SENSORS, select_sensor

In [ ]:
SURVEY_INDEX = 0
SURVEY_SPEED = 1.25    # m/s
SAVE_FIGURES = False   # write the figures to FIGURE_DIR

survey = load_survey(SURVEY_INDEX, SURVEY_SPEED)
survey

## Spectrograms

In [ ]:
plt.figure(figsize=(12, 8))
_, _, _, im = plt.specgram(select_sensor(survey, 3)["Total field [nT]"], NFFT=512, Fs=FS, noverlap=0,
                           mode="psd", cmap="magma")
cbar = plt.colorbar(im)
cbar.set_label("Power Spectral Density [dB]", fontsize=14)
cbar.ax.tick_params(labelsize=12)

limit = [0, 100]
plt.ylabel("Frequency [Hz]", fontsize=14)
plt.xlabel("Time [s]", fontsize=14)
plt.ylim(limit)
plt.yticks(np.arange(limit[0], limit[1] + 1, 2), fontsize=12)
plt.xticks(fontsize=12)
plt.grid(True, linestyle="--", linewidth=0.4, alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
freq_limit = [0, 100]
fig, axes = plt.subplots(5, 1, figsize=(8.0, 11.0), sharex=True)
plt.subplots_adjust(hspace=0.4)

for ax, sensor in zip(axes, SENSORS):
    signal = select_sensor(survey, sensor, on_track=True)["Total field [nT]"].values
    _, _, _, im = ax.specgram(signal, NFFT=1024, Fs=FS, noverlap=0, mode="psd", cmap="magma")
    ax.set_ylim(freq_limit)
    ax.set_yticks(np.arange(freq_limit[0], freq_limit[1] + 1, 20))
    ax.set_title(f"Sensor {sensor}", fontsize=12, loc="left", pad=6)
    ax.tick_params(labelsize=9)
    ax.grid(True, linestyle="--", linewidth=0.4, alpha=0.5)

fig.text(0.04, 0.5, "Freq [Hz]", va="center", rotation="vertical", fontsize=12)
axes[-1].set_xlabel("Time [s]", fontsize=12)
cbar_ax = fig.add_axes([0.92, 0.15, 0.015, 0.7])
fig.colorbar(im, cax=cbar_ax, label="Power Spectral Density [dB]")
plt.tight_layout(rect=[0.05, 0.05, 0.9, 0.97])

if SAVE_FIGURES:
    FIGURE_DIR.mkdir(exist_ok=True)
    fig.savefig(FIGURE_DIR / f"Flight_{SURVEY_INDEX + 1}_spectrograms.png", dpi=300, bbox_inches="tight")
plt.show()

## Sensor offsets (2 Hz lowpass, on-track)

In [ ]:
lowpassed = cf.lowpass_sensors(survey, "Total field [nT]", cutoff=2)
offsets = cf.sensor_offsets(lowpassed, reference_sensor=4)
print("Offsets:", {k: round(v, 2) for k, v in offsets.items()})

palette = sns.color_palette("pastel", n_colors=5)
plt.figure(figsize=(12, 6))
for sensor, signal in lowpassed.items():
    plt.plot(signal, label=f"Sensor {sensor}", color=palette[sensor - 1], alpha=0.8)
plt.xlabel("Sample")
plt.ylabel("Total field [nT]")
plt.legend(loc="upper right")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
components = ["Mag-X [nT]", "Mag-Y [nT]", "Mag-Z [nT]"]
palette = sns.color_palette("pastel", n_colors=5)

fig, axes = plt.subplots(3, 1, figsize=(8, 11), sharex=True)
plt.subplots_adjust(hspace=0.4)

for ax, comp in zip(axes, components):
    for sensor, signal in cf.lowpass_sensors(survey, comp, cutoff=2).items():
        ax.plot(signal, label=f"Sensor {sensor}", color=palette[sensor - 1], linewidth=0.8, alpha=0.9)
    ax.set_ylabel(comp, fontsize=12)
    ax.set_title(f"{comp} per Sensor", fontsize=14, loc="left")
    ax.grid(True, linestyle="--", alpha=0.5)
    ax.tick_params(labelsize=10)

axes[-1].set_xlabel("Sample index", fontsize=12)
axes[-1].legend(fontsize=10, loc="upper right")
plt.tight_layout(rect=[0.05, 0.05, 0.95, 0.97])
plt.show()

## Highpass noise (> 2 Hz, on-track)

In [ ]:
palette = sns.color_palette("pastel", n_colors=5)
fig, axes = plt.subplots(5, 1, figsize=(8, 11), sharex=True)

for ax, (sensor, res) in zip(axes, cf.highpass_noise(survey, cutoff=2).items()):
    ax.plot(res["noise"], label=f"Avg: {res['mean_abs']:.2f} nT\nStd: {res['std']:.2f}",
            color=palette[sensor - 1], alpha=0.6, linewidth=1.5)
    ax.plot(res["trend"], color="black", linestyle="--", linewidth=1)
    ax.axhline(res["std"], color="gray", linestyle=":", linewidth=2)
    ax.axhline(-res["std"], color="gray", linestyle=":", linewidth=2)

    y_pad = 2 * np.max(np.abs(res["noise"]))
    ax.set_ylim(-y_pad, y_pad)
    ax.set_title(f"Sensor {sensor}", fontsize=12, loc="left")
    ax.legend(fontsize=9, loc="upper left", bbox_to_anchor=(0.7, 1), frameon=False)
    ax.grid(True)
    if sensor != 5:
        ax.tick_params(labelbottom=False)

fig.text(0.5, 0.025, "Time (sample)", ha="center", fontsize=14)
fig.text(0.03, 0.5, "Noise [nT]", va="center", rotation="vertical", fontsize=14)
plt.tight_layout(rect=[0.05, 0.05, 0.9, 0.97])
plt.show()

## Harmonic bands

In [ ]:
# FIR design and band analysis of one harmonic for one sensor (add windows to compare them)
WINDOWS = ["kaiser"]   # e.g. "hann", "blackman", "hamming", "bartlett"
RIPPLE_DB = 60
n, sensor = 1, 1
center = cf.HARMONIC_BASE * n
lowcut, highcut = center - 0.5, center + 0.5

signal = select_sensor(survey, sensor, on_track=True)["Total field [nT]"].iloc[cf.TRIM:-cf.TRIM]
results = cf.compare_windows(signal, lowcut, highcut, windows=WINDOWS, transition_width=0.4, ripple_db=RIPPLE_DB)

fig, axs = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle(f"Interference Analysis (f = {center:.1f} Hz)", y=1.02, fontsize=20)
for name, res in results.items():
    axs[0, 0].plot(res["w"], np.abs(res["h"]), label=name)
    axs[0, 1].plot(res["w"], 20 * np.log10(np.maximum(np.abs(res["h"]), 1e-10)), label=name)
    axs[1, 0].plot(res["envelope"], label=name)
    axs[1, 1].plot(res["inst_freq"], label=name)

axs[0, 0].set_title("Bandpass Frequency Response", fontsize=16)
axs[0, 0].set_ylabel("Gain", fontsize=14)
axs[0, 0].axvline(lowcut, color="r", linestyle="--", alpha=0.3)
axs[0, 0].axvline(highcut, color="r", linestyle="--", alpha=0.3)

axs[0, 1].set_title("Stopband Attenuation", fontsize=16)
axs[0, 1].set_ylabel("Gain (dB)", fontsize=14)
axs[0, 1].axhline(-RIPPLE_DB, color="r", linestyle="--", alpha=0.5)
axs[0, 1].set_ylim(-RIPPLE_DB - 20, 5)

axs[1, 0].set_title("Amplitude Envelope", fontsize=16)
axs[1, 0].set_ylabel("Amplitude (nT)", fontsize=14)

axs[1, 1].set_title(f"Instantaneous Frequency Sensor {sensor}", fontsize=16)
axs[1, 1].set_ylabel("Frequency (Hz)", fontsize=14)
axs[1, 1].set_ylim(lowcut - 5, highcut + 5)

for ax in axs[0]:
    ax.set_xlim(lowcut - 5, highcut + 5)
    ax.set_xlabel("Freq (Hz)", fontsize=14)
for ax in axs[1]:
    ax.set_xlabel("Time (samples)", fontsize=14)
for ax in axs.flat:
    ax.grid(True)
    ax.legend(fontsize=12)
    ax.tick_params(axis="both", labelsize=12)

plt.tight_layout(rect=[0.03, 0.03, 0.97, 0.97])
fig.subplots_adjust(top=0.92)
plt.show()

In [ ]:
# Instantaneous frequency of the 20 Hz band per sensor (on-track)
lowcut, highcut = 19.5, 20.5
bands = cf.band_per_sensor(survey, lowcut, highcut, on_track=True, transition_width=0.4, ripple_db=60)

fig, axes = plt.subplots(len(bands), 1, figsize=(8.0, 11.0), sharex=False)
fig.subplots_adjust(hspace=0.4)
for ax, (sensor, res) in zip(axes, bands.items()):
    ax.plot(res["inst_freq"], label=f"Sensor {sensor}", alpha=0.8, linewidth=1)
    ax.set_ylim(lowcut - 5, highcut + 5)
    ax.set_yticks(np.arange(np.round(lowcut - 4.2), np.round(highcut + 4.8), 1))
    ax.set_ylabel("Freq [Hz]", fontsize=12)
    ax.set_title(f"Sensor {sensor}", fontsize=13, loc="left")
    ax.grid(True, linestyle="--", linewidth=0.5, alpha=0.4)
    ax.tick_params(labelsize=10)
axes[-1].set_xlabel("Time (samples)", fontsize=12)

plt.tight_layout(rect=[0.05, 0.05, 0.95, 0.97])
plt.show()

In [ ]:
# Envelope of the 20 Hz band, all sensors on one axis
lowcut, highcut = 19.5, 20.5
bands = cf.band_per_sensor(survey, lowcut, highcut, trim=cf.TRIM, transition_width=0.4, ripple_db=60)
pastel = sns.color_palette("pastel")

plt.figure(figsize=(10, 6))
for sensor, res in bands.items():
    plt.plot(res["envelope"], label=f"Sensor {sensor}", color=pastel[sensor - 1], linewidth=1, alpha=0.9)
plt.xlabel("Time (samples)", fontsize=14)
plt.ylabel("Amplitude (nT)", fontsize=14)
plt.title(f"{int((lowcut + highcut) / 2)} Hz Frequency Band Amplitude", fontsize=16, loc="left")
plt.legend(fontsize=11, loc="upper right", frameon=False)
plt.grid(True, linestyle="--", alpha=0.4)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Envelopes of the harmonics of the 20 Hz band
harmonics = cf.harmonic_bands(survey, base_freq=20, harmonics=[1, 2], transition_width=0.4, ripple_db=60)
pastel = sns.color_palette("pastel")

fig, axes = plt.subplots(len(harmonics), 1, figsize=(8.0, 11.0), sharex=False)
fig.subplots_adjust(hspace=0.5)
for ax, (n, bands) in zip(axes, harmonics.items()):
    for sensor, res in bands.items():
        ax.plot(res["envelope"], label=f"Sensor {sensor}", color=pastel[sensor - 1], linewidth=1, alpha=0.9)
    ax.set_ylabel("Amplitude (nT)", fontsize=12)
    ax.set_title(f"{20 * n} Hz Frequency Band Amplitude", fontsize=13, loc="left")
    ax.grid(True, linestyle="--", alpha=0.4)
    ax.tick_params(labelsize=10)
axes[-1].set_xlabel("Time (samples)", fontsize=12)

fig.suptitle("Amplitude Envelopes Across Sensors by Harmonic Band", fontsize=15, y=0.995)
plt.tight_layout(rect=[0.05, 0.05, 0.95, 0.97])
plt.show()

In [ ]:
# Envelope distributions of the 16.7 Hz harmonic family, truncated Gaussian fit
harmonic_numbers = [1, 2, 3, 4, 5]
distributions = cf.envelope_distributions(survey, harmonic_numbers)
pastel = sns.color_palette("pastel")

fig, axes = plt.subplots(len(SENSORS), len(harmonic_numbers), figsize=(4 * len(harmonic_numbers), 3.5 * len(SENSORS)),
                         squeeze=False)
fig.subplots_adjust(hspace=0.5, wspace=0.4)
for (sensor, n), res in distributions.items():
    ax = axes[sensor - 1, harmonic_numbers.index(n)]
    x = np.linspace(res["envelope"].min(), res["envelope"].max(), 100)
    ax.hist(res["envelope"], bins=50, density=True, alpha=0.6, color=pastel[sensor - 1], label="Data")
    ax.plot(x, res["dist"].pdf(x), "k--", linewidth=1.5, label=f"Fit μ={res['mu']:.2f}, σ={res['sigma']:.2f}")
    ax.set_title(f"Sensor {sensor} | {res['center']:.1f} Hz", fontsize=18)
    ax.tick_params(labelsize=18)
    ax.grid(True, linestyle="--", alpha=0.3)

fig.supxlabel("Amplitude [nT]", fontsize=24, x=0.6, y=0.05)
fig.supylabel("Density", fontsize=24)
plt.tight_layout(rect=[0.05, 0.05, 0.97, 0.93])
plt.show()

## ARMA model of a harmonic


In [ ]:
n = 2
SPLIT = 46000 

arma_survey = load_survey(4, SURVEY_SPEED)
model = cf.arma_harmonic_model(arma_survey, arma_survey, freq=cf.HARMONIC_BASE * n, sensor=3,
                               train_trim=(cf.TRIM, SPLIT), test_trim=(SPLIT, -cf.TRIM))
model["fit"].summary()

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(model["t"], model["actual"], label=f"Actual {cf.HARMONIC_BASE * n:.1f} Hz Signal", linewidth=1)
plt.plot(model["t"], model["synthetic"], label=r"Synthetic $\mathrm{ARMA} \cdot \sin(2\pi f t)$",
         alpha=0.4, linewidth=1, linestyle="--")
plt.title(f"Synthetic Harmonic vs Actual Bandpassed Signal\nRMSE = {model['rmse']:.4f}", fontsize=24)
plt.xlabel("Time (s)", fontsize=16)
plt.ylabel("Total Field (nT)", fontsize=16)
plt.legend(fontsize=18)
plt.grid(True)
plt.xticks(fontsize=16)
plt.yticks(fontsize=16)
plt.tight_layout()
plt.show()